# 8.13 — Positional encodings

Position encodings give attention a sense of before and after without giving up its parallel view of the whole sequence.

Self-attention compares token contents, so this lesson asks how the model knows which vector came first. Sinusoidal coordinates, RoPE rotations, ALiBi score biases, and learned tables each inject order in a different place.

Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 713
random.seed(SEED)
np.random.seed(SEED)


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - np.max(arr)
    exp = np.exp(shifted)
    return exp / exp.sum()


def accuracy(pred, gold):
    return sum(int(a == b) for a, b in zip(pred, gold)) / max(1, len(gold))


def show_table(rows, headers):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    print(" | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers)))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(value).ljust(widths[i]) for i, value in enumerate(row)))


def make_sequence_ladder(topic):
    if topic == "8.13":
        return [
            {"name": "D1 reversal", "sequences": [["dog", "bites"], ["bites", "dog"]], "labels": [1, 0], "max_pos": 2},
            {"name": "D2 clean order", "sequences": [["a", "then", "b"], ["b", "then", "a"], ["red", "before", "blue"], ["blue", "before", "red"], ["left", "right"]], "labels": [1, 0, 1, 0, 1], "max_pos": 5},
            {"name": "D3 agreement", "sequences": [["dogs", "chase", "cat"], ["cat", "chase", "dogs"], ["he", "likes", "tea"], ["tea", "likes", "he"], ["birds", "fly", "today"]], "labels": [1, 0, 1, 0, 1], "max_pos": 8},
            {"name": "D4 pair tasks", "sequences": [["subject", "verb", "object", "now"], ["object", "verb", "subject", "now"], ["query", "key", "value", "end"], ["value", "key", "query", "end"], ["first", "middle", "last", "stop"]], "labels": [1, 0, 1, 0, 1], "max_pos": 10},
            {"name": "D5 beyond learned", "sequences": [["start"] + ["filler"] * n + ["end"] for n in [4, 8, 12, 16, 20]], "labels": [1, 1, 1, 1, 1], "max_pos": 22},
        ]
    if topic == "8.14":
        return [
            {"name": "D1 mask toy", "T": 32, "w": 2, "evidence": [(19, 0)], "global_tokens": []},
            {"name": "D2 local deps", "T": 24, "w": 2, "evidence": [(4, 3), (8, 6), (12, 11), (16, 15), (20, 18)], "global_tokens": []},
            {"name": "D3 global random", "T": 40, "w": 2, "evidence": [(30, 0), (35, 1), (21, 18), (10, 8), (39, 3)], "global_tokens": [0, 1]},
            {"name": "D4 document snippets", "T": 64, "w": 3, "evidence": [(50, 2), (60, 4), (32, 31), (12, 10), (45, 5)], "global_tokens": [0, 2, 4]},
            {"name": "D5 far evidence", "T": 96, "w": 3, "evidence": [(90, 0), (88, 6), (70, 9), (44, 43), (30, 27)], "global_tokens": [0, 6, 9]},
        ]
    if topic == "8.15":
        return [
            {"name": "D1 logits", "rows": [[3.0, 2.0, 0.5]], "target": ["A"]},
            {"name": "D2 clean rows", "rows": [[3, 1, 0], [0, 3, 1], [1, 0, 3], [4, 2, 0], [0, 4, 2]], "target": list("ABCAB")},
            {"name": "D3 traps", "rows": [[2.1, 2.0, 0], [1.8, 1.7, 1.6], [3, 2.9, 0], [2, 2, 1.9], [0, 2.2, 2.1]], "target": list("BACCB")},
            {"name": "D4 candidates", "rows": [[2.5, 2.4, 0.1], [0.5, 2.0, 1.9], [2.2, 2.1, 2.0], [3.0, 0.2, 0.1], [0.1, 2.8, 2.7]], "target": list("ABABB")},
            {"name": "D5 long generation", "rows": [[2.0, 1.9, 0.2], [1.5, 1.4, 2.2], [2.1, 2.0, 0.4], [0.4, 2.2, 2.1], [1.9, 1.8, 0.7], [0.5, 2.4, 2.3], [2.3, 2.2, 0.2]], "target": list("ACABABB")},
        ]
    if topic == "8.16":
        return [
            {"name": "D1 lesson pair", "cand": "the cat sat", "ref": "the cat slept", "probs": [0.5, 0.25, 0.25]},
            {"name": "D2 paraphrases", "pairs": [("small cat sleeps", "small cat sleeps"), ("red bird flies", "red bird sings"), ("quick dog runs", "dog runs quickly"), ("bright moon rises", "moon rises bright"), ("blue car stops", "blue bus stops")]},
            {"name": "D3 predicate order", "pairs": [("the cat chased mouse", "the mouse chased cat"), ("doctor treats patient", "doctor helps patient"), ("team wins final", "team loses final"), ("user resets password", "password resets user"), ("model answers question", "model answers query")]},
            {"name": "D4 summaries", "pairs": [("sales rose in june", "sales rose in june after ads"), ("server failed at noon", "server recovered at noon"), ("policy changed for users", "policy changed for all users"), ("campaign reached creators", "campaign reached many creators"), ("video views increased", "video watch time increased")]},
            {"name": "D5 long refs", "pairs": [("the model answered safely with citations", "the model answered correctly with citations"), ("the summary covered three facts and omitted risk", "the summary covered three facts and named risk"), ("translation kept names but changed tense", "translation kept names and preserved tense"), ("caption mentions dog running near beach", "caption mentions dog walking near beach"), ("assistant refused unsafe request politely", "assistant refused unsafe request clearly")]},
        ]
    if topic == "8.17":
        return [
            {"name": "D1 Ada", "tokens": ["Ada", "works", "at", "OpenAI"], "tags": ["B-PER", "O", "O", "B-ORG"]},
            {"name": "D2 clean entities", "examples": [(["Ada", "joined", "OpenAI"], ["B-PER", "O", "B-ORG"]), (["Lin", "visits", "Paris"], ["B-PER", "O", "B-LOC"]), (["DeepMind", "hired", "Ilya"], ["B-ORG", "O", "B-PER"]), (["Mira", "met", "Sam"], ["B-PER", "O", "B-PER"]), (["Google", "opened", "office"], ["B-ORG", "O", "O"])]},
            {"name": "D3 adjacent padding", "examples": [(["Ada", "Bob", "spoke"], ["B-PER", "B-PER", "O"]), (["New", "York", "Labs"], ["B-LOC", "I-LOC", "B-ORG"]), (["Eve", "from", "IBM", "arrived"], ["B-PER", "O", "B-ORG", "O"]), (["Paris", "Paris", "team"], ["B-LOC", "B-ORG", "O"]), (["Zed", "Corp", "LLC"], ["B-ORG", "I-ORG", "I-ORG"])]},
            {"name": "D4 snippets", "examples": [(["Dr", "Chen", "at", "Mayo"], ["B-PER", "I-PER", "O", "B-ORG"]), (["Acme", "bought", "Beta"], ["B-ORG", "O", "B-ORG"]), (["Nina", "left", "Berlin"], ["B-PER", "O", "B-LOC"]), (["OpenAI", "San", "Francisco"], ["B-ORG", "B-LOC", "I-LOC"]), (["Raj", "from", "UN"], ["B-PER", "O", "B-ORG"])]},
            {"name": "D5 long multi", "examples": [(["Ada", "Lovelace", "worked", "with", "Babbage", "Labs"], ["B-PER", "I-PER", "O", "O", "B-ORG", "I-ORG"]), (["Maya", "from", "OpenAI", "visited", "London"], ["B-PER", "O", "B-ORG", "O", "B-LOC"]), (["IBM", "and", "Google", "met", "in", "Paris"], ["B-ORG", "O", "B-ORG", "O", "O", "B-LOC"]), (["Dr", "Lee", "joined", "Mayo", "Clinic"], ["B-PER", "I-PER", "O", "B-ORG", "I-ORG"]), (["Sam", "Altman", "visited", "New", "York"], ["B-PER", "I-PER", "O", "B-LOC", "I-LOC"])]},
        ]
    return [
        {"name": "D1 morphology", "tokens": ["walk", "walked", "walking"], "tags": ["VERB", "VERB", "VERB"]},
        {"name": "D2 clean POS", "examples": [(["dogs", "walk"], ["NOUN", "VERB"]), (["they", "walked"], ["NOUN", "VERB"]), (["walking", "helps"], ["NOUN", "VERB"]), (["red", "dogs", "run"], ["ADJ", "NOUN", "VERB"]), (["cats", "sleep"], ["NOUN", "VERB"])]},
        {"name": "D3 ambiguous", "examples": [(["walk", "home"], ["VERB", "NOUN"]), (["the", "walk"], ["DET", "NOUN"]), (["walking", "tour"], ["ADJ", "NOUN"]), (["we", "duck"], ["NOUN", "VERB"]), (["duck", "soup"], ["NOUN", "NOUN"])]},
        {"name": "D4 tagged lines", "examples": [(["bright", "birds", "sing"], ["ADJ", "NOUN", "VERB"]), (["users", "liked", "ads"], ["NOUN", "VERB", "NOUN"]), (["fast", "models", "serve"], ["ADJ", "NOUN", "VERB"]), (["the", "running", "joke"], ["DET", "ADJ", "NOUN"]), (["creators", "earned", "money"], ["NOUN", "VERB", "NOUN"])]},
        {"name": "D5 agreement", "examples": [(["the", "dogs", "were", "walking"], ["DET", "NOUN", "VERB", "VERB"]), (["a", "walk", "was", "short"], ["DET", "NOUN", "VERB", "ADJ"]), (["walking", "brands", "sell"], ["ADJ", "NOUN", "VERB"]), (["names", "ending", "ing", "confuse"], ["NOUN", "VERB", "NOUN", "VERB"]), (["past", "users", "walked"], ["ADJ", "NOUN", "VERB"])]},
    ]



def sinusoidal_position(pos, d=4):
    values = []
    for i in range(d // 2):
        denom = 10000 ** (2 * i / d)
        values.append(math.sin(pos / denom))
        values.append(math.cos(pos / denom))
    return np.array(values)


def rope_rotate(vec, pos):
    theta = float(pos)
    rot = np.array([[math.cos(theta), -math.sin(theta)], [math.sin(theta), math.cos(theta)]])
    return rot @ np.array(vec, dtype=float)


def alibi_bias(distance, slope=-0.5):
    return slope * abs(distance)


def add_positions(tokens, kind="sinusoidal", learned_table=None):
    reps = []
    for pos, token in enumerate(tokens):
        base = np.array([1.0, 0.0]) if token <= tokens[0] else np.array([0.0, 1.0])
        if kind == "learned":
            if learned_table is None or pos >= len(learned_table):
                raise IndexError("learned position table is out of range")
            reps.append(base + np.array(learned_table[pos]))
        elif kind == "rope":
            reps.append(rope_rotate(base, pos))
        else:
            reps.append(base + sinusoidal_position(pos, 2))
    return np.vstack(reps)


def order_sensitive_predict(seq, kind="sinusoidal", max_learned=10):
    if kind == "content":
        return 1
    if kind == "learned" and len(seq) > max_learned:
        return 0
    return int(seq[0] != "bites" and seq[0] != "object" and seq[0] != "value" and seq[0] != "blue" and seq[0] != "tea")


## The concept, built once: add position before attention

The lesson's sinusoid is $PE(pos,2i)=\sin(pos/10000^{2i/d})$ and $PE(pos,2i+1)=\cos(pos/10000^{2i/d})$. We first show why content alone cannot distinguish a two-token reversal.

In [ ]:

original = np.array([[1, 0], [0, 1]])
reversed_tokens = np.array([[0, 1], [1, 0]])
original_multiset = original[np.lexsort((original[:, 1], original[:, 0]))]
reversed_multiset = reversed_tokens[np.lexsort((reversed_tokens[:, 1], reversed_tokens[:, 0]))]
difference = original_multiset - reversed_multiset
print(difference)
assert difference.tolist() == [[0, 0], [0, 0]]


RoPE applies a real 2D rotation $R_\theta$ to both query and key pairs before their dot product. ALiBi instead adds a distance bias directly to attention scores, and the sign must make far positions more negative: $-2.5\lt -0.5$.

In [ ]:

row = sinusoidal_position(pos=1, d=4)
rotated_query = rope_rotate([1.0, 0.0], pos=1)
rotated_key = rope_rotate([0.0, 1.0], pos=1)
near_bias = alibi_bias(1, slope=-0.5)
far_bias = alibi_bias(5, slope=-0.5)
learned_row = [0, 0.2]
print("sinusoid", np.round(row, 6))
print("rope dot", round(float(rotated_query @ rotated_key), 6))
print("biases", far_bias, near_bias)
assert far_bias == -2.5
assert near_bias == -0.5
assert far_bias < near_bias
assert learned_row == [0, 0.2]


## The dataset ladder

Family F7 is inline: D1 is the lesson reversal, then clean order, agreement, pair classification, and D5 extrapolation beyond a learned table.

In [ ]:

ladder = make_sequence_ladder("8.13")
for rung in ladder:
    sample = rung["sequences"][0]
    print(rung["name"], "n=", len(rung["sequences"]), "max_pos=", rung["max_pos"], "sample=", sample)


In [ ]:

rows = []
metrics = []
for rung in ladder:
    preds = [order_sensitive_predict(seq, kind="sinusoidal") for seq in rung["sequences"]]
    score = accuracy(preds, rung["labels"])
    metrics.append(score)
    rows.append([rung["name"], len(rung["sequences"]), round(score, 3)])
show_table(rows, ["rung", "items", "accuracy"])


In [ ]:

fig, axes = plt.subplots(1, 6, figsize=(15, 2.4))
for ax, rung in zip(axes[:5], ladder):
    length = min(rung["max_pos"], 12)
    mat = np.vstack([sinusoidal_position(pos, 4) for pos in range(length)])
    ax.imshow(mat, aspect="auto", cmap="viridis")
    ax.set_title(rung["name"].split()[0])
    ax.set_xlabel("dim")
    ax.set_ylabel("pos")
axes[5].plot(range(1, 6), metrics, marker="o")
axes[5].set_ylim(0, 1.05)
axes[5].set_title("accuracy")
axes[5].set_xlabel("rung")
fig.tight_layout()


## Pitfall on D5: learned range and ALiBi sign

A learned table only has allocated rows; extrapolating past it fails. A flipped ALiBi sign rewards far tokens instead of discouraging them.

In [ ]:

d5 = ladder[-1]
learned_table = [[0.0, 0.0] for _ in range(10)]
try:
    add_positions(d5["sequences"][-1], kind="learned", learned_table=learned_table)
    learned_ok = True
except IndexError:
    learned_ok = False
wrong_far = alibi_bias(5, slope=0.5)
right_far = alibi_bias(5, slope=-0.5)
fixed_preds = [order_sensitive_predict(seq, kind="sinusoidal") for seq in d5["sequences"]]
print("learned extrapolated", learned_ok)
print("wrong far bias", wrong_far)
print("right far bias", right_far)
print("fixed D5 accuracy", accuracy(fixed_preds, d5["labels"]))
assert learned_ok is False
assert wrong_far > 0
assert right_far == -2.5


## Evaluate it + Practice

- Metric: order-sensitive accuracy; no-skill content-only baseline predicts the same label for reversals.
- Sanity check: reversing two distinct tokens must change the positional representation.
- Ablation: remove position information and D1 collapses to the zero multiset difference.
- Failure signal: learned positions throw or silently clamp beyond their trained length.

Practice: implement a relative-position bias matrix for D3.

Practice: compare rotating only queries with rotating both queries and keys.